# Data Wrangling – Methodology

**Objective:** Clean the Falcon 9 launch data, engineer the binary `Class` (landing success) label, and produce the analysis-ready dataset.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`

## Note
This notebook is self-contained: instead of depending on an uploaded `dataset_part_1.csv`,
it fetches the same data directly from the SpaceX API. Since the live SpaceX API
(`api.spacexdata.com`) can be unreliable or unreachable, this notebook first tries a
static mirror of the same data, and falls back to IBM's pre-built dataset if both the
live API and the mirror are unavailable — so it can be run standalone in any fresh
Colab session without manual file uploads.

## Methodology
1. Fetch Falcon 9 launch data (API / mirror / fallback CSV).
2. Inspect missing values per column.
3. Compute launch-site value counts, orbit-type value counts, and landing-outcome (`Outcome`) value counts.
4. Define which outcomes count as a **successful landing** (`bad_outcomes`) vs. failure.
5. Create the binary target column `Class` (1 = landed, 0 = did not land).
6. Save the wrangled dataset as `dataset_part_2_clean.csv` for EDA and modeling.


## Part A: Build the dataset (self-fetch from SpaceX API, with fallback)

### Step 1: Request rocket launch data from the SpaceX API

In [1]:
# Step 1: Fetch raw launch data + helper functions to resolve linked IDs
import requests
import pandas as pd
import numpy as np
import datetime
import time

BASE = "https://api.spacexdata.com/v4"

# NOTE: the live SpaceX API (api.spacexdata.com) is often unreliable and can
# return a Cloudflare "525 SSL handshake failed" error even with a valid
# User-Agent/session. To make the results reproducible, we first try IBM's
# own static mirror of the same `launches/past` response, and only fall back
# to the live API if that mirror is unreachable.
STATIC_LAUNCHES_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Capstone-Project/1.0",
    "Accept": "application/json"
})


def get_json(url, retries=3, wait=3):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            last_err = f"status {resp.status_code}"
        except requests.exceptions.RequestException as e:
            last_err = str(e)
        print(f"  Attempt {attempt} failed ({last_err}), retrying in {wait}s...")
        time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {url} after {retries} attempts: {last_err}")


# Lookup lists that the helper functions below will populate
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial = [], [], [], []
Longitude, Latitude = [], []


def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = get_json(f"{BASE}/rockets/{x}")
            BoosterVersion.append(response['name'])


def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = get_json(f"{BASE}/launchpads/{x}")
            LaunchSite.append(response['name'])
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])


def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = get_json(f"{BASE}/payloads/{load[0]}")
            PayloadMass.append(response.get('mass_kg'))
            Orbit.append(response.get('orbit'))


def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = get_json(f"{BASE}/cores/{core['core']}")
            Block.append(response.get('block'))
            ReusedCount.append(response.get('reuse_count'))
            Serial.append(response.get('serial'))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(f"{core['landing_success']} {core['landing_type']}")
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])


# Try the static mirror first (reliable), fall back to the live API
try:
    raw_json = get_json(STATIC_LAUNCHES_URL, retries=2, wait=2)
    print("Fetched launches from static mirror. Records:", len(raw_json))
except RuntimeError as e:
    print("Static mirror failed, falling back to live API:", e)
    spacex_url = f"{BASE}/launches/past"
    raw_json = get_json(spacex_url, retries=5, wait=3)
    print("Fetched launches from live API. Records:", len(raw_json))

data = pd.json_normalize(raw_json)

# Keep only the columns we need, drop multi-core / multi-payload launches (single-stack Falcon 9 flights)
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print("Raw launches after single-core/single-payload filter:", data.shape)


Fetched launches from static mirror. Records: 107
Raw launches after single-core/single-payload filter: (94, 7)


### Step 2: Resolve booster/launchpad/payload/core lookups and assemble the flat DataFrame

In [2]:
# Step 2: Call the helper functions to populate lookup lists, then assemble the flat DataFrame

# NOTE: as of June 2026 the r-spacex/SpaceX-API project (which powers
# api.spacexdata.com) was archived and its origin now returns a permanent
# Cloudflare "525" error on every endpoint - including /rockets/{id},
# /launchpads/{id}, /payloads/{id} and /cores/{id}. No amount of retrying
# fixes this, it is simply gone for good.
#
# We still attempt the live lookups first (in case the API is ever restored),
# but if they fail we fall back to IBM's own pre-built copy of this exact
# dataset - it has the identical columns this cell is trying to build.
FALLBACK_DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)

try:
    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)

    launch_dict = {
        'FlightNumber': list(data['flight_number']),
        'Date': list(data['date']),
        'BoosterVersion': BoosterVersion,
        'PayloadMass': PayloadMass,
        'Orbit': Orbit,
        'LaunchSite': LaunchSite,
        'Outcome': Outcome,
        'Flights': Flights,
        'GridFins': GridFins,
        'Reused': Reused,
        'Legs': Legs,
        'LandingPad': LandingPad,
        'Block': Block,
        'ReusedCount': ReusedCount,
        'Serial': Serial,
        'Longitude': Longitude,
        'Latitude': Latitude
    }

    data_falcon9 = pd.DataFrame(launch_dict)

    # Derive the binary landing-success target column ('class') from Outcome text
    bad_outcomes = {'None None', 'None ASDS', 'None RTLS', 'False ASDS', 'False Ocean', 'False RTLS'}

    print("Assembled DataFrame from live API lookups. Shape:", data_falcon9.shape)

except RuntimeError as e:
    print("Live API lookups unavailable (api.spacexdata.com is down):", e)
    print("Falling back to IBM's pre-built dataset_part_1.csv ...")
    data_falcon9 = pd.read_csv(FALLBACK_DATASET_URL)
    print("Loaded fallback dataset. Shape:", data_falcon9.shape)

data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv")
data_falcon9.head()


  Attempt 1 failed (status 525), retrying in 3s...
  Attempt 2 failed (status 525), retrying in 3s...
  Attempt 3 failed (status 525), retrying in 3s...
Live API lookups unavailable (api.spacexdata.com is down): Failed to fetch https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb after 3 attempts: status 525
Falling back to IBM's pre-built dataset_part_1.csv ...
Loaded fallback dataset. Shape: (90, 17)
Saved dataset_part_1.csv


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [3]:
# Filter to Falcon 9 only (drop Falcon 1 launches)
data_falcon9 = data_falcon9[data_falcon9['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print("Filtered shape:", data_falcon9.shape)
data_falcon9.isnull().sum()


Filtered shape: (90, 17)


,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


In [4]:
# Fill missing PayloadMass with the column mean
mean_mass = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(mean_mass)

# This becomes our working dataframe for the rest of this notebook (Part B)
df = data_falcon9.copy()
df.to_csv('dataset_part_1.csv', index=False)
print("Built dataset with shape:", df.shape)
df.head()


Built dataset with shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


### Missing values

In [5]:
df.isnull().sum() / len(df) * 100


,0
FlightNumber,0.000000
Date,0.000000
BoosterVersion,0.000000
PayloadMass,0.000000
Orbit,0.000000
LaunchSite,0.000000
Outcome,0.000000
Flights,0.000000
GridFins,0.000000
Reused,0.000000


In [6]:
df.dtypes


,0
FlightNumber,int64
Date,object
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


### Launch site counts

In [7]:
df['LaunchSite'].value_counts()


,count
LaunchSite,
CCAFS SLC 40,55
KSC LC 39A,22
VAFB SLC 4E,13


### Orbit counts

In [8]:
df['Orbit'].value_counts()


,count
Orbit,
GTO,27
ISS,21
VLEO,14
PO,9
LEO,7
SSO,5
MEO,3
HEO,1
ES-L1,1


### Landing outcome counts

In [9]:
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes


,count
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


In [10]:
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)


0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


In [11]:
# Outcome strings look like 'True ASDS', 'False Ocean', 'None None', etc.
# (landing_success + ' ' + landing_type). A landing only counts as a
# SUCCESS when landing_success is True — everything else (False or None)
# is a failed/no-attempt landing. Matching on the 'True' prefix instead of
# hardcoded value_counts() positions means this stays correct no matter
# how many distinct outcomes exist or what order they come in.
bad_outcomes = {outcome for outcome in landing_outcomes.keys() if not outcome.startswith('True')}
bad_outcomes


{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

### Create the binary `Class` label

In [12]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]
df['Class'] = landing_class
df[['Outcome', 'Class']].head(10)


,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1
8,None None,0
9,None None,0


In [13]:
success_rate = df['Class'].mean()
print(f"Overall landing success rate: {success_rate:.2%}")


Overall landing success rate: 66.67%


In [14]:
df.to_csv('dataset_part_2_clean.csv', index=False)
print("Saved dataset_part_2_clean.csv with shape:", df.shape)


Saved dataset_part_2_clean.csv with shape: (90, 18)


## Summary
- Loaded the raw API dataset and audited missing values (mainly in `LandingPad`, expected for expendable/failed landings).
- Reviewed launch-site and orbit-type frequency distributions.
- Mapped each distinct `Outcome` string into `bad_outcomes` (failure) vs. success — based on whether the landing_success flag was `True` — then derived the binary `Class` column used as the ML target for the rest of the project.
- Exported `dataset_part_2_clean.csv`.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`
